# IR Inspection

This notebook is a thin review surface over the canonical MotifML IR.
It loads either a single `.ir.json` file or one selected member from an IR corpus
directory and renders deterministic summaries and review tables.

In [ ]:
from __future__ import annotations

import os
from pathlib import Path

from IPython.display import SVG, Markdown, display

from motifml.ir.review_tables import (
    build_control_event_rows,
    build_onset_note_tables,
    build_structure_summary,
    build_voice_lane_onset_tables,
    load_ir_document_record,
    render_control_event_rows_markdown,
    render_onset_note_table_markdown,
    render_structure_summary_markdown,
    render_voice_lane_onset_table_markdown,
)
from motifml.ir.review_visualizations import (
    render_control_timeline_svg,
    render_note_relations_svg,
    render_timeline_plot_svg,
    render_voice_lane_ladder_svg,
)


def _find_repo_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").exists():
            return candidate
    raise RuntimeError("Could not locate the MotifML repository root.")


repo_root = _find_repo_root(Path.cwd())
default_input_path = (
    repo_root
    / "tests"
    / "fixtures"
    / "ir"
    / "golden"
    / "ensemble_polyphony_controls.ir.json"
)
input_path = Path(os.environ.get("MOTIFML_IR_INPUT_PATH", default_input_path))
relative_path = os.environ.get("MOTIFML_IR_MEMBER")

record = load_ir_document_record(input_path, relative_path)
summary = build_structure_summary(record.document)
voice_lane_tables = build_voice_lane_onset_tables(record.document)
onset_note_tables = build_onset_note_tables(record.document)
control_rows = build_control_event_rows(record.document)

display(
    Markdown(
        "\n".join(
            (
                "## Loaded Document",
                "",
                f"- Review identity: `{record.relative_path}`",
                f"- Input path: `{input_path.as_posix()}`",
                f"- Corpus member: `{relative_path}`"
                if relative_path
                else "- Corpus member: single document input",
            )
        )
    )
)

In [ ]:
display(Markdown("## Structure Rollup"))
display(Markdown(render_structure_summary_markdown(summary)))

display(Markdown("## Voice-Lane Onset Tables"))
for table in voice_lane_tables:
    display(Markdown(render_voice_lane_onset_table_markdown(table)))

display(Markdown("## Onset Note Tables"))
for table in onset_note_tables:
    display(Markdown(render_onset_note_table_markdown(table)))

display(Markdown("## Control Events"))
display(Markdown(render_control_event_rows_markdown(control_rows)))

display(Markdown("## Review Visualizations"))
display(SVG(render_timeline_plot_svg(record.document)))
display(SVG(render_voice_lane_ladder_svg(record.document)))
display(SVG(render_note_relations_svg(record.document)))
display(SVG(render_control_timeline_svg(record.document)))